# R4-N1 LR-A and BiGRU-A: waveform-to-decision parity (Colab)
Checks that the two development bundles reproduce the saved raw recording scores and decisions from **real audio**: canonical shard audio goes through each bundle's scorer (tier 1) and, re-encoded as a 16 kHz mono PCM16 WAV, through the shared `analyze` path (tier 2). It replays all 1,300 operating recordings plus 100 each from the selection and calibration roles. It trains and fits nothing, changes no bundle, and reads no A test recording.

Owner: Subhan (bundles built locally on 16 November 2026). Sync first (same paths on Drive): `src/echoproof/models/r4_n1_raw_bundles.py`, `scripts/r4_n1_raw_parity.py`, `tests/test_r4_n1_raw_bundles.py`, `configs/r4_n1_calibration_a.json`, `src/echoproof/evaluation/r4_n1.py`; `bundles/r4_n1_lr_A_raw_threshold/` and `bundles/r4_n1_bigru_A_raw_threshold/`; the runs `runs/r4/r4_n1_lr/` and `runs/r4/r4_n1_bigru/` (with `handoff/bundle_packaging.json`); the R4-M1 exports and models under `runs/r4/r4_m1_a_v1_*`; `data/manifests/cohort_v1.parquet`; the shards `data/processed/cohort_v1/`.

Tolerances on recording scores: LR 1e-6, BiGRU 1e-5 (the exports' own reload tolerances were 1e-12 and 1e-6). A failed run is reported, not rerun with a looser tolerance. Each parity report is written once; a second run for the same bundle is refused.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
OWNER = 'Subhan'
SHARDS = ROOT / 'data/processed/cohort_v1'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, str(ROOT / 'src'))
import torch, numpy, librosa, sklearn, pyarrow
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__, '| numpy', numpy.__version__,
      '| librosa', librosa.__version__, '| scikit-learn', sklearn.__version__, '| pyarrow', pyarrow.__version__)
required = ['configs/r4_n1_calibration_a.json', 'scripts/r4_n1_raw_parity.py', 'src/echoproof/models/r4_n1_raw_bundles.py', 'src/echoproof/evaluation/r4_n1.py',
            'bundles/r4_n1_lr_A_raw_threshold/bundle.json', 'bundles/r4_n1_bigru_A_raw_threshold/bundle.json',
            'runs/r4/r4_n1_lr/handoff/bundle_packaging.json', 'runs/r4/r4_n1_bigru/handoff/bundle_packaging.json',
            'runs/r4/r4_m1_a_v1_lr_export/predictions_recordings_operating.parquet', 'runs/r4/r4_m1_a_v1_bigru_export/predictions_recordings_operating.parquet',
            'data/manifests/cohort_v1.parquet']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing: {missing}'
assert SHARDS.is_dir(), 'Set SHARDS to the canonical shard folder'
for model in ('lr', 'bigru'):
    assert not (ROOT / f'runs/r4/r4_n1_{model}/handoff/waveform_parity.json').exists(), f'{model}: a parity report already exists; keep it or move it aside deliberately'
print('inputs present; no earlier parity report')

## Tests (nothing is replayed until they pass)
The bundle, loader and tamper tests. A failure or a skip stops the notebook. (`RealBundleTests` runs here because the built bundles are present on Drive.)

In [ ]:
import re
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', '-rs', 'tests/test_r4_n1_raw_bundles.py'], text=True, capture_output=True)
print(check.stdout[-2500:], check.stderr[-1500:])
assert check.returncode == 0, 'Resolve the test failures first'
assert not re.findall(r'^SKIPPED', check.stdout, flags=re.M), 'Unexpected skips: the built bundles must be present'
print('tests passed; skips: none')

## LR-A parity (CPU)
Handcrafted features from the waveform chunks, the saved LR model, mean chunk probability, the serialized threshold.

In [ ]:
def parity(model, device):
    result = subprocess.run([sys.executable, 'scripts/r4_n1_raw_parity.py', '--model', model, '--root', str(ROOT), '--shard-dir', str(SHARDS), '--owner', OWNER, '--device', device],
                            text=True, capture_output=True)
    print(result.stdout[-3500:], result.stderr[-2500:])
    return result.returncode
lr_rc = parity('lr', 'cpu')
print('LR-A parity exit code:', lr_rc)

## BiGRU-A parity (GPU)
Waveform chunks through the log-mel front end and the two-layer BiGRU of the packaged checkpoint, then the same rule. Use a T4 runtime.

In [ ]:
bigru_rc = parity('bigru', 'cuda' if torch.cuda.is_available() else 'cpu')
print('BiGRU-A parity exit code:', bigru_rc)

## Summary
Reads the two reports; it does not change them. If either failed, send the printed output and stop: do not rerun with a changed tolerance.

In [ ]:
for model in ('lr', 'bigru'):
    path = ROOT / f'runs/r4/r4_n1_{model}/handoff/waveform_parity.json'
    if not path.is_file():
        print(model, 'no report'); continue
    r = json.loads(path.read_text(encoding='utf-8'))
    print(model, '| passed:', r['passed'], '| recordings:', r['recordings'], r['roles'], '| tolerance:', r['tolerance'], '| device:', r['device'])
    for tier, t in r['tiers'].items():
        print('  ', tier, {k: t[k] for k in ('scored', 'unscored', 'max_raw_difference', 'max_score_difference', 'decision_flips', 'unexplained_flips', 'within_tolerance') if k in t})
    print('   tier1 vs tier2:', r.get('tier1_vs_tier2'), '| flips:', len(r['flips']), '| near threshold:', len(r['near_threshold']), '| chunk-count mismatches:', len(r['chunk_count_mismatches']))